# Snippet from Cookbook.md


In [ ]:
"""Minimal interactive REPL over a real CompitumRouter."""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import yaml

from compitum.cli import _load_constraints, _toy_models
from compitum.boundary import BoundaryAnalyzer
from compitum.coherence import CoherenceFunctional
from compitum.constraints import ReflectiveConstraintSolver
from compitum.control import LyapunovController
from compitum.energy import SymbolicFreeEnergy
from compitum.metric import SymbolicManifoldMetric
from compitum.pgd import RegexPromptExtractor
from compitum.predictors import CalibratedPredictor
from compitum.router import CompitumRouter


def build_router(seed: int = 12345) -> CompitumRouter:
    dcfg = yaml.safe_load(Path("configs/router_defaults.yaml").read_text())
    D = int(dcfg["metric"]["D"])
    rank = int(dcfg["metric"]["rank"])
    delta = float(dcfg["metric"]["delta"])
    models = _toy_models(D)
    np.random.seed(seed)
    rng = np.random.default_rng(seed)
    X_demo = rng.standard_normal((512, D))
    predictors = {}
    for m in models:
        yq = 0.6 + 0.1 * np.tanh(X_demo @ (m.center / np.linalg.norm(m.center) + 1e-8))
        yt = 0.5 + 0.5 * np.abs(X_demo @ np.ones(D) / np.sqrt(D))
        yc = 0.2 + 0.4 * np.abs(X_demo @ (np.arange(D) / D))
        pq = CalibratedPredictor(); pq.fit(X_demo, yq)
        pt = CalibratedPredictor(); pt.fit(X_demo, yt)
        pc = CalibratedPredictor(); pc.fit(X_demo, yc)
        predictors[m.name] = {"quality": pq, "latency": pt, "cost": pc}
    metrics = {m.name: SymbolicManifoldMetric(D, rank, delta) for m in models}
    coherence = CoherenceFunctional(k=500)
    A, b = _load_constraints(Path("configs/constraints_us_default.yaml"))
    solver = ReflectiveConstraintSolver(A, b)
    boundary = BoundaryAnalyzer(gap_threshold=0.05, entropy_threshold=0.65, sigma_threshold=0.12)
    controller = LyapunovController()
    energy = SymbolicFreeEnergy(dcfg["alpha"], dcfg["beta_t"], dcfg["beta_c"], dcfg["beta_d"], dcfg["beta_s"])
    pgd = RegexPromptExtractor()
    return CompitumRouter(
        models, predictors, solver, coherence, boundary, controller, pgd, metrics, energy,
        update_stride=int(dcfg["update_stride"]),
    )


def main() -> None:
    router = build_router()
    print("compitum REPL -- type a prompt and press Enter ('quit' to exit)")
    while True:
        try:
            prompt = input("> ").strip()
        except EOFError:
            break
        if not prompt or prompt.lower() == "quit":
            break
        cert = json.loads(router.route(prompt).to_json())
        b = cert["boundary"]
        print(f"  model={cert['model']} utility={cert['utility']:.4f} "
              f"gap={b['utility_gap']:.4f} entropy={b['entropy']:.4f} "
              f"is_boundary={b['is_boundary']}")


if __name__ == "__main__":
    main()
